# Ridge pool model
Global Ridge (L2) trained on pooled data. Features: same as LSTM (log-return lags, volume_lag_1, RSI, MACD) + VIX 5-day SMA, **vix_velocity**, **vix_momentum**, month_sin/cos, Fear & Greed change. Multi-output: 7-step **log returns**; convert to price via `p0 * exp(cumsum(pred_log_returns))`. **Inputs scaled with StandardScaler before fit**.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor

REPO_ROOT = Path.cwd().parent.parent
BACKEND_DIR = REPO_ROOT / "backend"
sys.path.insert(0, str(BACKEND_DIR))
sys.path.insert(0, str(Path.cwd()))

from _pool_common import (
    load_pool_data,
    build_pooled_train_stack,
    compute_metrics_averaged_over_windows,
    metrics_to_parquet,
    fetch_cnn_fear_greed_index,
    TEST_SIZE,
    FORECAST_HORIZON,
    ROLLING_STEP,
    MIN_TRAIN_STACK,
    ARTIFACTS_DIR,
    TICKERS,
)

LAG_RETURNS = 5
RSI_PERIOD = 7
MACD_FAST, MACD_SLOW, MACD_SIGNAL = 12, 26, 9
# Load best alpha from random search if available (run random search cell first to generate)
import json
_ridge_path = ARTIFACTS_DIR / "best_ridge_params.json"
if _ridge_path.exists():
    with open(_ridge_path) as f:
        _ridge = json.load(f)
    RIDGE_ALPHA = _ridge["alpha"]
else:
    RIDGE_ALPHA = 1.0  # L2 regularization strength

In [2]:
def _rsi(series: pd.Series, period: int) -> pd.Series:
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = (-delta).clip(lower=0)
    avg_gain = gain.ewm(alpha=1 / period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / period, adjust=False).mean()
    rs = avg_gain / np.where(avg_loss != 0, avg_loss, 1e-10)
    return 100 - (100 / (1 + rs))


def build_feature_df(grp: pd.DataFrame):
    """Features: LSTM-style (log-return lags, volume_lag_1, RSI, MACD) + VIX 5-day SMA, vix_velocity, vix_momentum, month sin/cos, Fear & Greed change. Target = next 7 log returns. All scaled before fit."""
    df = grp.sort_values("timestamp").copy()
    df["close"] = df["close"].astype(float)
    df["return"] = df["close"].pct_change()
    df["log_return"] = np.log(df["close"] / df["close"].shift(1))
    for i in range(1, LAG_RETURNS + 1):
        df[f"ret_lag_{i}"] = df["log_return"].shift(i)
    if "volume" in df.columns:
        df["volume_lag_1"] = df["volume"].astype(float).shift(1)
    else:
        df["volume_lag_1"] = np.nan
    df["rsi"] = _rsi(df["close"], RSI_PERIOD)
    ema_fast = df["close"].ewm(span=MACD_FAST, adjust=False).mean()
    ema_slow = df["close"].ewm(span=MACD_SLOW, adjust=False).mean()
    df["macd_line"] = ema_fast - ema_slow
    df["macd_signal"] = df["macd_line"].ewm(span=MACD_SIGNAL, adjust=False).mean()
    if "vix" in df.columns:
        vix = df["vix"].astype(np.float64)
        df["vix_sma_5"] = vix.shift(1).rolling(5).mean()
        df["vix_velocity"] = vix.diff(1)
        df["vix_momentum"] = vix - df["vix_sma_5"]
    else:
        df["vix_sma_5"] = np.nan
        df["vix_velocity"] = np.nan
        df["vix_momentum"] = np.nan
    df["month"] = pd.to_datetime(df["timestamp"]).dt.month
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    if "fear_greed" not in df.columns:
        df["fear_greed"] = 50.0
    else:
        df["fear_greed"] = df["fear_greed"].fillna(50.0)
    df["fear_greed_lag_1"] = df["fear_greed"].shift(1)
    df["fear_greed_lag_5"] = df["fear_greed"].shift(5)
    df["fear_greed_change"] = df["fear_greed_lag_1"] - df["fear_greed_lag_5"]
    for h in range(1, FORECAST_HORIZON + 1):
        df[f"target_{h}"] = df["return"].shift(-h)
    feature_cols_lstm = [f"ret_lag_{i}" for i in range(1, LAG_RETURNS + 1)] + ["volume_lag_1", "rsi", "macd_line", "macd_signal"]
    feature_cols_xgb = ["vix_sma_5", "vix_velocity", "vix_momentum", "month_sin", "month_cos", "fear_greed_change"]
    feature_cols = feature_cols_lstm + feature_cols_xgb
    target_cols = [f"target_{h}" for h in range(1, FORECAST_HORIZON + 1)]
    base_cols = ["timestamp", "close", "return", "log_return"] + feature_cols + target_cols
    out = df[[c for c in base_cols if c in df.columns]].copy()
    return out.dropna(), feature_cols, target_cols


def train_global_ridge(stacked: pd.DataFrame, horizon: int):
    """Train one Ridge (MultiOutputRegressor) on pooled data. Returns dict for predict_ridge_global."""
    pooled = build_pooled_train_stack(stacked, TEST_SIZE, MIN_TRAIN_STACK)
    if pooled.empty:
        return None
    feat_dfs = []
    for sym in pooled["symbol"].unique():
        grp = pooled[pooled["symbol"] == sym].copy()
        try:
            feat_df, feature_cols, target_cols = build_feature_df(grp)
        except Exception:
            continue
        if len(feat_df) < MIN_TRAIN_STACK + horizon:
            continue
        feat_dfs.append(feat_df)
    if not feat_dfs:
        return None
    pooled_feat = pd.concat(feat_dfs, ignore_index=True)
    X = pooled_feat[feature_cols].values.astype(np.float64)
    y = pooled_feat[target_cols].values.astype(np.float64)
    scaler = StandardScaler()
    X_s = scaler.fit_transform(X)
    ridge = MultiOutputRegressor(Ridge(alpha=RIDGE_ALPHA, random_state=42))
    ridge.fit(X_s, y)
    return {"model": ridge, "scaler": scaler, "feature_cols": feature_cols}


def predict_ridge_global(context_df: pd.DataFrame, horizon: int, global_ridge: dict) -> list:
    """Predict 7 price steps using pre-trained global Ridge (model outputs log returns; compound to price)."""
    if global_ridge is None:
        return []
    try:
        feat_df, feature_cols, _ = build_feature_df(context_df)
    except Exception:
        return []
    if len(feat_df) < 1:
        return []
    X = feat_df[feature_cols].values.astype(np.float64)
    X_s = global_ridge["scaler"].transform(X)
    last_row = X_s[-1:]
    pred_log_returns = global_ridge["model"].predict(last_row).ravel()
    p0 = float(context_df["close"].iloc[-1])
    prices = p0 * np.exp(np.cumsum(pred_log_returns))
    return [float(p) for p in prices[:horizon]]

In [3]:
stacked = load_pool_data(with_vix=True, with_volume=True)
symbol_start = pd.to_datetime(stacked["timestamp"]).min().strftime("%Y-%m-%d")
fear_greed_df = fetch_cnn_fear_greed_index(start_date=symbol_start)
if not fear_greed_df.empty:
    stacked["date"] = pd.to_datetime(stacked["timestamp"]).dt.normalize()
    fear_greed_df["date"] = pd.to_datetime(fear_greed_df["timestamp"]).dt.normalize()
    stacked = stacked.merge(fear_greed_df[["date", "fear_greed"]], on="date", how="left")
    stacked["fear_greed"] = stacked["fear_greed"].ffill().bfill()
    stacked = stacked.drop(columns=["date"])
print(stacked.groupby("symbol").size())
stacked.head(10)

symbol
AAPL     1256
AMZN     1256
GOOGL    1256
JNJ      1256
JPM      1256
MSFT     1256
NVDA     1256
SPY      1256
WMT      1256
XOM      1256
dtype: int64


,timestamp,symbol,close,volume,vix,fear_greed
0,2021-03-01,AAPL,127.790001,116307900,23.350000,58.52
1,2021-03-02,AAPL,125.120003,102260900,24.100000,50.32
2,2021-03-03,AAPL,122.059998,112966300,26.670000,43.84
3,2021-03-04,AAPL,120.129997,178155000,28.570000,37.80
4,2021-03-05,AAPL,121.419998,153766600,24.660000,41.52
5,2021-03-08,AAPL,116.360001,154376600,25.469999,39.08
6,2021-03-09,AAPL,121.089996,129525800,24.030001,43.36
7,2021-03-10,AAPL,119.980003,111943300,22.559999,45.56
8,2021-03-11,AAPL,121.959999,103026500,21.910000,50.48
9,2021-03-12,AAPL,121.029999,88105100,20.690001,53.72


In [4]:
# 20-iteration Random Search: Mean MAE across 7 horizons; save best params for stack notebooks
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import make_scorer
import json

def mean_mae_multioutput(y_true, y_pred):
    """Mean MAE over all 7 horizon columns (higher is worse; we use neg_mean_mae for scoring)."""
    return np.mean(np.abs(y_true - y_pred))

neg_mean_mae_scorer = make_scorer(mean_mae_multioutput, greater_is_better=False)

pooled = build_pooled_train_stack(stacked, TEST_SIZE, MIN_TRAIN_STACK)
feat_dfs = []
for sym in pooled["symbol"].unique():
    grp = pooled[pooled["symbol"] == sym].copy()
    try:
        feat_df, feature_cols, target_cols = build_feature_df(grp)
    except Exception:
        continue
    if len(feat_df) < MIN_TRAIN_STACK + FORECAST_HORIZON:
        continue
    feat_dfs.append(feat_df)
pooled_feat = pd.concat(feat_dfs, ignore_index=True)
pooled_feat = pooled_feat.sort_values("timestamp").reset_index(drop=True)
X_rs = pooled_feat[feature_cols].values.astype(np.float64)
y_rs = pooled_feat[target_cols].values.astype(np.float64)

pipe_ridge = Pipeline([
    ("scaler", StandardScaler()),
    ("multi", MultiOutputRegressor(Ridge(random_state=42)))
])
param_dist_ridge = {
    "multi__estimator__alpha": [1e-5, 5e-5, 1e-4, 5e-4, 0.001, 0.005, 0.01, 0.1, 1.0, 10.0],
}
tscv = TimeSeriesSplit(n_splits=5)
search_ridge = RandomizedSearchCV(
    pipe_ridge, param_dist_ridge, n_iter=20, cv=tscv, scoring=neg_mean_mae_scorer,
    random_state=42, n_jobs=-1, verbose=1
)
search_ridge.fit(X_rs, y_rs)

best_ridge = {k.replace("multi__estimator__", ""): float(v) for k, v in search_ridge.best_params_.items()}
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
with open(ARTIFACTS_DIR / "best_ridge_params.json", "w") as f:
    json.dump(best_ridge, f, indent=2)
RIDGE_ALPHA = best_ridge["alpha"]
print("Best Mean MAE (CV):", -search_ridge.best_score_)
print("Best RIDGE_ALPHA:", RIDGE_ALPHA)
print("Saved to", ARTIFACTS_DIR / "best_ridge_params.json")

c:\capstone_project_unfc\env\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 10 is smaller than n_iter=20. Running 10 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best Mean MAE (CV): 0.013246955133991652
Best RIDGE_ALPHA: 10.0
Saved to C:\capstone_project_unfc\model\experiments-pool\artifacts\best_ridge_params.json


In [5]:
global_ridge = train_global_ridge(stacked, FORECAST_HORIZON)
print("Global Ridge trained on", len(build_pooled_train_stack(stacked, TEST_SIZE, MIN_TRAIN_STACK)), "pooled train rows.")

Global Ridge trained on 11960 pooled train rows.


In [6]:
# Ridge coefficients per horizon (each of 21 outputs has its own linear model)
if global_ridge is not None:
    multi = global_ridge["model"]
    cols = global_ridge["feature_cols"]
    coefs = np.array([multi.estimators_[h].coef_ for h in range(len(multi.estimators_))])  # (21, n_features)
    intercepts = np.array([multi.estimators_[h].intercept_ for h in range(len(multi.estimators_))])
    df_coef = pd.DataFrame(coefs.T, index=cols, columns=[f"h{h+1}" for h in range(coefs.shape[0])])
    print("Ridge coefficients per horizon (rows=features, columns=horizon):")
    display(df_coef)
    mean_coef = df_coef.mean(axis=1)
    mean_abs_coef = df_coef.abs().mean(axis=1)
    summary = pd.DataFrame({"mean_coef": mean_coef, "mean_abs_coef": mean_abs_coef}).sort_values("mean_abs_coef", ascending=False)
    print("\nSummary (mean coefficient and mean |coefficient| across horizons):")
    display(summary)
    print("\nIntercepts per horizon:", intercepts)
else:
    print("No trained model (global_ridge is None). Run the training cell above first.")

Ridge coefficients per horizon (rows=features, columns=horizon):


,h1,h2,h3,h4,h5,h6,h7
ret_lag_1,0.000053,-5.400782e-04,0.000028,-0.000020,0.000007,-0.000411,-0.000126
ret_lag_2,-0.000693,7.074952e-05,0.000227,0.000203,-0.000330,-0.000240,0.000411
ret_lag_3,-0.000011,1.885953e-04,0.000285,-0.000180,-0.000213,0.000370,-0.000309
ret_lag_4,0.000196,1.695115e-04,-0.000129,-0.000087,0.000362,-0.000320,0.000292
ret_lag_5,0.000201,-1.730078e-04,-0.000137,0.000453,-0.000358,0.000281,-0.000022
volume_lag_1,0.000670,4.559580e-04,0.000555,0.000546,0.000533,0.000461,0.000492
rsi,0.000375,8.232707e-05,0.000128,0.000361,0.000650,0.000510,-0.000213
macd_line,-0.001053,-8.764167e-04,-0.001011,-0.001358,-0.001472,-0.001238,-0.000375
macd_signal,0.000597,5.880073e-04,0.000720,0.000943,0.001027,0.000832,0.000296
vix_sma_5,0.000231,1.526425e-04,0.000141,0.000070,0.000013,0.000045,0.000050



Summary (mean coefficient and mean |coefficient| across horizons):


,mean_coef,mean_abs_coef
macd_line,-0.001055,0.001055
macd_signal,0.000715,0.000715
volume_lag_1,0.000531,0.000531
vix_momentum,0.000201,0.000504
vix_velocity,0.000149,0.000403
rsi,0.000271,0.000332
ret_lag_2,-0.000050,0.000311
ret_lag_5,0.000035,0.000232
ret_lag_3,0.000019,0.000222
ret_lag_4,0.000069,0.000222



Intercepts per horizon: [0.00090018 0.00089802 0.00089863 0.00090482 0.00090883 0.00091221
 0.00091079]


In [7]:
model_name = "ridge"
all_preds = []
for sym in TICKERS:
    grp = stacked[stacked["symbol"] == sym].copy()
    if grp.empty:
        continue
    grp = grp.sort_values("timestamp").reset_index(drop=True)
    prices = grp.set_index("timestamp")["close"].astype(float).dropna()
    n = len(prices)
    if n < TEST_SIZE + MIN_TRAIN_STACK:
        continue
    split_idx = n - TEST_SIZE
    test_index = prices.index[split_idx:]
    test_values = prices.values[split_idx:]
    preds = []
    window_ix = 0
    start = 0
    while start + FORECAST_HORIZON <= TEST_SIZE:
        context_cols = ["timestamp", "close", "vix"] + [c for c in ["volume", "fear_greed"] if c in grp.columns]
        context_df = grp.iloc[: split_idx + start][context_cols].copy()
        if len(context_df) < MIN_TRAIN_STACK:
            start += ROLLING_STEP
            continue
        price_list = predict_ridge_global(context_df, FORECAST_HORIZON, global_ridge)
        if not price_list or len(price_list) < FORECAST_HORIZON:
            start += ROLLING_STEP
            window_ix += 1
            continue
        for h in range(FORECAST_HORIZON):
            idx = start + h
            ts = test_index[idx]
            y_true = float(test_values[idx])
            y_pred = float(price_list[h])
            preds.append({"timestamp": ts, "y_true": y_true, "y_pred": y_pred, "window_ix": window_ix})
        window_ix += 1
        start += ROLLING_STEP
    if preds:
        pred_df = pd.DataFrame(preds)
        pred_df["symbol"] = sym
        all_preds.append(pred_df)

pred_stack = pd.concat(all_preds, ignore_index=True) if all_preds else pd.DataFrame(columns=["timestamp", "y_true", "y_pred", "window_ix", "symbol"])
print(pred_stack.groupby("symbol").size() if not pred_stack.empty else "No predictions.")
pred_stack.head()

symbol
AAPL     56
AMZN     56
GOOGL    56
JNJ      56
JPM      56
MSFT     56
NVDA     56
SPY      56
WMT      56
XOM      56
dtype: int64


,timestamp,y_true,y_pred,window_ix,symbol
0,2025-12-02,286.190002,283.785109,0,AAPL
1,2025-12-03,284.149994,284.274781,0,AAPL
2,2025-12-04,280.700012,284.566539,0,AAPL
3,2025-12-05,278.779999,284.581972,0,AAPL
4,2025-12-08,277.890015,284.783997,0,AAPL


In [8]:
metrics_rows = []
for sym in pred_stack["symbol"].unique():
    sub = pred_stack[pred_stack["symbol"] == sym]
    m = compute_metrics_averaged_over_windows(sub)
    metrics_rows.append({"model": model_name, "symbol": sym, **m})
m_overall = compute_metrics_averaged_over_windows(pred_stack)
metrics_rows.append({"model": model_name, "symbol": "overall", **m_overall})

metrics_df = pd.DataFrame(metrics_rows)
print(metrics_df.to_string())
metrics_to_parquet(metrics_rows, ARTIFACTS_DIR / "metrics_ridge_pool.parquet")
print("Saved:", ARTIFACTS_DIR / "metrics_ridge_pool.parquet")

    model   symbol        MAE       RMSE    MAPE_%
0   ridge     AAPL   6.745000   7.628856  2.561743
1   ridge     MSFT  11.684341  12.743408  2.661695
2   ridge    GOOGL   7.933243   8.952793  2.494583
3   ridge     AMZN   9.105188  10.067068  4.053022
4   ridge      JPM   7.500820   8.240217  2.392005
5   ridge      JNJ   4.155133   4.597753  1.863983
6   ridge      WMT   2.565596   2.888127  2.134160
7   ridge      SPY   6.612131   7.441535  0.964681
8   ridge      XOM   4.592345   4.950328  3.355270
9   ridge     NVDA   4.122951   4.828846  2.264367
10  ridge  overall   6.501675   8.475328  2.474551
Saved: C:\capstone_project_unfc\model\experiments-pool\artifacts\metrics_ridge_pool.parquet
